# Lesson 4.3: Geoparsing in Python


**🎬 Video:** [Lesson 4.3: Geoparsing in Python](#)

## Overview

In this lesson, we'll take text data we scraped from Reddit and:

1. **Extract Locations**: Use a sophisticated geoparser to find and resolve geographic references
2. **Visualize on Maps**: Create interactive maps showing retrieved locations
3. **Find Errors**: Mapping the results makes it easier to see where the model ran into trouble

Geoparsing is the process of extracting locations from a dataset and mapping those locations to real world coordinates. This is very complex and the actual setup and syntax are beyond the scope of an introductory course. What is more, running the actual code for a large dataset requires a lot of computing power. Instead, this lesson will show you what a sophisticated machine learning library like this can do, but also focus on the limits on of machine learning models. Ultimately, the output generated by computation needs to be corrected and interpreted by humans. This is particularly true of models that try to extract places in human speech. 

While humans can have a specific understanding of a place that specificity is rarely expressed in speech. More plainly, places and spaces in everyday conversation can be quite "fuzzy." Consider the following examples:

1. "Meet me at DHall"
2. "Saw this downtown, last week"
3. "When I was at JMU I had classes at Reed Hall"

These are all examples of places on the Reddit thread that JMU students could infer from context, but a machine learning model would not be able to figure out without a significant amount of fine-tuning. In the case of the first, a building like DHall might be listed in a database of locations, but without knowing that this is a building on the JMU campus it would be hard for a computer to make an educated guess as to where this might be. Second, "downtown" could mean anything, but in the Reddit thread it means Harrisonburg. Finally, "Reed Hall" is now Keezell hall and only a JMU student or alum would know this difference. In theory, there is a way to "train" machine learning models to be more accurate, but this still requires a lot of human work. In fact, for a relatively small dataset like this Reddit thread it would take more effort to train the model than it would to just fill in the missing information manually. 

Which is why for this lesson we will be manually correcting a spreadsheet. This is conceptually similar to the way training machine learning models works: we are getting better results through manual human intervention to clarify the ambiguities of human language.

> 👉 **Note:** *This lesson is made possible by the [geoparser](https://github.com/dguzh/geoparser) library created by Diego Gomes.*

---

## 📖 1 Follow Along — How Geoparsing Works (and Where It Fails)

You do not need to write or modify any code in this section. Read through the examples and focus on understanding what the geoparser can and cannot do, and why.

### Overview 

A geoparser identifies place names in text and resolves them to real geographic coordinates. **Resolving** a place name means finding its actual latitude and longitude. The technical term for this is **toponym disambiguation**.

For example, we found "Harrisonburg" in our Reddit data, but which one? There are two:

- **Harrisonburg, VA** — ~55,000 residents, home of JMU
- **Harrisonburg, LA** — a few hundred residents

During disambiguation, the geoparser makes an educated guess based on context: other nearby places mentioned in the text, how often the place appears in the corpus, and the size of its population. Since, Harrisonburg, LA is smaller than Harrisonburg, VA it is weighted as less likely to occur.

> 👉 **Note:** *Google Maps uses the same technique. Searching for "London" generally results in London, UK appears first, even if London, KY is geographically closer to you.*


### 1.1 Watching the Model Run

The geoparser works as a two-step pipeline on every piece of text:

1. **Recognize** — A transformer-based language model reads the text and identifies which words are place names. This is the same NER step from [lesson 4.2](./lesson_4_2_using_ner.ipynb), but using a more powerful model.
2. **Resolve** — A second model looks at the surrounding context and picks the most likely match from the [GeoNames](https://www.geonames.org/) database — an open-access catalog of over 13 million geographic features, each with coordinates, population, and administrative information.

The three cells below show the code that runs the geoparser. **Do not run them** — the geoparser requires a ~13 GB database and a 500 MB language model that are not installed in this environment. Read the code to understand the structure, then watch the animation below it to see what actually happens when it runs.

> 👉 **Note:** *This is a common situation in computational research: tools that are too resource-intensive to run interactively. Understanding the code is still valuable — you need to know what inputs it expects and what outputs it produces.*


In [ ]:
# ⚠️ DO NOT RUN — for reading only. The geoparser is not installed in this environment.

from geoparser import Geoparser

# Initialize the pipeline: NER model → geographic transformer → GeoNames gazetteer
geo = Geoparser(
    spacy_model="en_core_web_trf",
    transformer_model="dguzh/geo-all-distilroberta-v1",
    gazetteer="geonames",
)

# Parse a list of sentences — each returns a doc with a .toponyms list
test_sentences = [
    "I traveled from New York to Richmond, Virginia last summer.",
    "The battle took place near Harrisonburg in the Shenandoah Valley.",
    "There is no direct bus that goes from Hburg to Lynchburg so he had to pick me up from Lex to drive me to his house.",
    "Don't let my post detour you from exploring JMU as an option.",
]

docs = geo.parse(test_sentences)

for i, doc in enumerate(docs):
    print(f"\nSentence {i+1}: \"{test_sentences[i]}\"")
    for toponym in doc.toponyms:
        loc = toponym.location
        if loc:
            print(f"  📍 '{toponym.text}' → {loc.get("name")} ({loc.get("latitude")}, {loc.get("longitude")}) — {loc.get("country_name")}")
        else:
            print(f"  ❓ '{toponym.text}' — recognized but could not be resolved")


Here is what running those cells actually looks like. It takes about 40 seconds to analyze about four sentences.

<img src="../lesson_assets/videos/output_gif/geoparser-demo.gif" alt="Screen recording of the geoparser processing test sentences, showing a progress bar and elapsed time" width="700">

Let's look at the output a bit more carefully:

<img src="../lesson_assets/images/output-geoparsing.png" alt="Output after geoparsing">

> 💡 **Reflection:** Look at the output above. What mistakes does the geoparser make in each sentence? Try to think like a computer for a moment — why would it make those specific errors? The last sentence was taken directly from the Reddit sample. How many mistakes can you find?


### 1.2 Where the Geoparser Fails

The output above reveals six distinct failure modes, all visible in just four sentences.

---

**Failure 1 — Too Many Places (False Positive)**

*"I traveled from New York to **Richmond, Virginia** last summer."*

The geoparser tags both `Richmond` and `Virginia` as separate locations. But `Virginia` here is not an independent destination, it just clarifies *which* Richmond. A human reader immediately understands this; the model does not. This inflates the location count and can skew any analysis that treats every tagged place as equally meaningful.

---

**Failure 2 — Wrong scale (Imprecise Resolution)**

*"The battle took place near Harrisonburg in the **Shenandoah Valley**."*

The geoparser finds `Shenandoah Valley` but can't properly resolve the scale. Is this meant to be a building, a city, a region or a state? It infers that this is referring to Community Christian School of the Shenandoah Valley instead of the Shenandoah Valley, which is a region.

---

**Failure 3 — Informal Abbreviation (False Negative)**

*"There is no direct bus that goes from **Hburg** to Lynchburg…"*

The model completely misses `Hburg`. It was trained on formal text — news, Wikipedia — where this abbreviation does not appear. Any shorthand that is locally understood but not in the training data will be silently skipped.

---

**Failure 4 — Ambiguous Abbreviation (False Negative or Wrong Entity Type)**

*"…so he had to pick me up from **Lex** to drive me to his house."*

The model does not identify `Lex` as a place at all. Short abbreviations like this are easily confused with a person's name (Lex Luthor, for instance). Without broader context the model cannot tell whether `Lex` is a city or a character, so it skips it entirely.

---

**Failure 5 — Hyperlocal Context (Unknowable Without Thread)**

*"…to drive me to **his house**."*

`his house` is a location — it is where this person was staying in Lynchburg. No geoparser can resolve this. Doing so would require reading the entire thread to establish who `he` is, where he lives, and then looking that up in a database. This kind of reference can only be interpreted by a human who has followed the full conversation.

---

**Failure 6 — Wrong Disambiguation (False Resolution)**

*"Don't let my post detour you from exploring **JMU** as an option."*

The model correctly identifies `JMU` as a place name and finds it in GeoNames, but resolves it to **Jiamusi, China**. Why? `JMU` is the IATA airport code for Jiamusi Dongjiao Airport in Heilongjiang province. Without any other geographic signal in the sentence, the model cannot distinguish the university acronym from the airport code, and confidently places a dot in northeastern China. This happened for all 40 sentences in the dataset containing `JMU`!

> 💡 **Reflection:** Look at the six failure types above. Which one do you think would be the most difficult to fix — and could you write a rule to catch it automatically, or would every case need human judgment?

### 1.3 Why Your Review Matters

None of these errors are unique to this geoparser. These are the fundamental limitations of any automated system trained on formal text. The model performs well on clearly written, formally named places. It struggles with informal language, ambiguous names, and non-geographic references.

Your pre-processed data files are already in your group folder. **You do not need to run the geoparsing pipeline.** Skip ahead to **Part 2** to load the data and visualize it on a map.

> 👉 **Note:** *If you are curious about how the geoparsing pipeline was actually run — the full working code, the batch processing loop, and how the output was saved — see [Lesson 4.5 - Technical Reference](lesson_4_5_technical_reference_geoparser.ipynb). That notebook is for reference only and is not required reading.*



---

## 2 Visualizing Spatial Relations

It is hard to detect location errors in a large spreadsheet with hundreds of locations. The location errors are much easier to see on a map. `plotly` can use another technology called `mapbox` to quickly plot georeferenced data on a map. This gives us better insight into the geoparsed file. In the script below, the geoparsed file is slightly modified by adding a column for the number of times that a particular location occurs in the Reddit posts. Then the data is filtered to exclude anything below 4 mentions. This helps us fix the major issues without worrying about low-level errors. Finally, the data is passed through to Plotly and turned into a map. In our previous `px.bar()` and `px.line` charts we set the variables `x` and `y`. We are doing the same thing here, but setting latitude (`lat`) and longitude `lon` on a map. Conceptually, this is very similar.

- **latitude** indicates position on earth north-to-south (90.0 to -90.0)
- **longitude** indicates position on earth east-to-west (180 to -180)


<img src="../lesson_assets/images/lat-long-explanation.png" alt="latitude and longitude map" style="width: 50%;">


Remember that most locations in the thread will likely be in the northwest quadrant and the value should range from 0 to 90 for latitude and 0 to -180 for longitude. Harrisonburg is roughly 38 N, -78 W.



### 2.1 Map the Results

The cell below loads the geoparsed data, filters it, and renders the map. Change `COUNT` to control the minimum number of mentions a place needs before it appears on the map. A higher value can remove clutter, while a lower value gives better insight into where the geoparser succeeds and fails.


In [ ]:
COUNT = 4  # ← change this to show more or fewer locations on the map

import pandas as pd
import plotly.express as px

df_geoparsed = pd.read_csv("../data/JMU/JMU_geoparsed_long.csv")

def format_sentences(sentences, max_sentences=5, max_chars=120):
    unique = list(dict.fromkeys(sentences))[:max_sentences]
    return "<br>".join(
        f'• {s[:max_chars]}{"…" if len(s) > max_chars else ""}' for s in unique
    )

df_map = (
    df_geoparsed.groupby("place")
    .agg(
        latitude=("latitude", "first"),
        longitude=("longitude", "first"),
        count=("place", "size"),
        examples=("sentences", format_sentences),
    )
    .reset_index()
)

df_map = df_map[df_map["count"] >= COUNT].reset_index(drop=True)

fig = px.scatter_map(
    df_map,
    lat="latitude",
    lon="longitude",
    hover_name="place",
    size="count",
    custom_data=["count", "examples"],
    title=f"JMU Locations with {COUNT} or More Occurrences",
    zoom=3,
    height=600,
)

fig.update_traces(
    hovertemplate=(
        "<b>%{hovertext}</b><br>"
        "Mentions: %{customdata[0]}<br><br>"
        "<i>Example posts:</i><br>"
        "%{customdata[1]}"
        "<extra></extra>"
    )
)

fig.update_layout(
    map_style="carto-voyager",
    margin={"r": 0, "t": 50, "l": 0, "b": 0},
)

fig.show()


> 📊 **Output:** Each bubble on the map represents a place the geoparser resolved to coordinates. Bubble size reflects how often that place was mentioned. Hover over any bubble to see the place name, mention count, and sample sentences. Adjust `COUNT` and re-run to explore more or fewer locations.


> 💡 **Reflection:** Hover over the map locations to get a sense of what places the `geoparser` package found accurate and which need to be corrected. 
> - Are there any surprising misidentifications? Places that seem obvious to you, but are totally alien to the language model?
> - What can explain these errors?

---

## Lesson Summary

**Part 1 — How Geoparsing Works (and Where It Fails)**
- `geo.parse(text)` — passes a string to the geoparser and returns a list of candidate place records, each with coordinates and a confidence score
- A geoparser combines NER (to find place-name strings) with a gazetteer lookup (to convert names to coordinates); it fails on ambiguous names, abbreviations, and informal references
- Every parsed row must be reviewed because automated confidence scores do not catch context errors (e.g. "Madison" meaning a person, not a city)

**Part 2 — Visualizing Spatial Relations**
- `pd.read_csv('file.csv')` — reloads a previously saved geoparsed dataset
- `px.scatter_map(df, lat=..., lon=..., hover_name=...)` — plots each place as a dot on an interactive map using latitude and longitude columns

> 👉 **Note:** *The full batch geoparsing run takes several minutes. The pre-processed output used in Part 2 was already created for you.*

➡️ **Next:** [Lesson 4.4 — Collaborative Location Review](lesson_4_4_preparing_review_sheet.ipynb)